**Hierarchical Clustering** is an unsupervised algorithm that groups data points into a tree of nested clusters without requiring you to specify the number of clusters ($K$) upfront.

# Core Intuition (How It Works)

The most common approach is Agglomerative (Bottom-Up) clustering:

1. **Start**: Treat every data point as its own individual cluster ($N$ points = $N$ clusters).
2. **Compute**: Calculate the pairwise distance matrix between all clusters.
3. **Merge**: Find the two closest clusters based on a specified distance metric and linkage criterion, then combine them into a single cluster.
4. **Repeat**: Recalculate distances and repeat step 3 until all points are merged into one large root cluster.
5. **Cut**: Slice the resulting tree (dendrogram) horizontally at a desired height/distance threshold to retrieve $K$ clusters.

(Note: The opposite approach is Divisive (Top-Down), which starts with one giant cluster containing all points and recursively splits it into smaller clusters.)

# Mathematical Example

1. **Dataset**
    $A=1, B=2, C=6, D=8$

2. **Compute Initial Distance Matrix**

    |Cluster|A(1)|B(2)|C(6)|D(8)|
    |---|---|---|---|---|
    |$A$|$0$|$\mathbf{1}$|$5$|$7$|
    |$B$|$1$|$0$|$4$|$6$|
    |$C$|$5$|$4$|$0$|$2$|
    |$D$|$7$|$6$|$2$|$0$|


3. **Merge Closest Pair**
    $d(A, B) = \vert{}1 - 2\vert{} = 1$. Merge $A$ and $B$ into cluster $\{AB\}$.

4. **Update Distance Matrix**
    - Single Linkage means where the distance between two clusters is defined as the shortest distance between any single point in the first cluster and any single point in the second cluster.
    e.g. Distance from (AB) to C = Min(Distance A-C, Distance B-C) = Min(5, 4) = 4
    Distance from (AB) to D = Min(Distance A-D, Distance B-D) = Min(7, 6) = 6

    |Cluster|{AB}|C|D|
    |---|---|---|---|
    |{AB}|0|4|6|
    |C|4|0|2|
    |D|6|2|0|

5. **Repeat the Step 3 and 4**

    - **Merge Next Closest Pair**:
    $d(C, D) = 2$. Merge $C$ and $D$ into cluster $\{CD\}$.

    - **Update Distance Matrix**:
    $d(\{AB\}, \{CD\}) = \min(d(\{AB\}, C), d(\{AB\}, D)) = \min(4, 6) = 4$

    |Cluster|{AB}|{CD}|
    |---|---|---|
    |{AB}|0|4|
    |{CD}|4|0|

    - **Final Merge**
    Merge $\{AB\}$ and $\{CD\}$ into a single root cluster $\{ABCD\}$.

6. **Dendogram**
```
Merge Height
   ▲
 4 ┼       ┌──────────────────────────┐
   │       |                          |      
 3 ┼       |                          |        
   │       |                          |  
 2 ┼       |                    ┌─────┴─────┐         
   │       |                    |           |     
 1 ┼  ┌────┴─────┐              │           │     
   │  │          │              │           │    
───┴──┴──────────┴──────────────┴───────────┴────────►
      A (1)     B (2)          C (6)      D (8)   (Data Points) 
```

7. **Picking the Best K**

    - Look for the longest vertical lines in your dendrogram that don't have any horizontal cross-bars crossing them.
    - In this case, the biggest empty veritcal gap is between height 2 and 4 (gap of 2), so slice it through the middle (at height =3) gives you **K=2**. 



# Concept You must Know

- **Dendrogram**: A tree-like diagram that illustrates the sequence of merges or splits and the exact distances at which each merge occurred.

- **Linkage Criteria:** Rules that define how distance is measured between two multi-point clusters:

    - **Single Linkage:** Minimum distance between any point in Cluster A and any point in Cluster B (sensitive to noise/chaining).

    - **Complete Linkage:** Maximum distance between any point in Cluster A and any point in Cluster B (produces compact clusters).

    - **Average Linkage:** Average distance between all pairs of points across Cluster A and Cluster B.

    - **Ward's Linkage:** Minimizes the increase in total within-cluster variance after merging.

- **Cophenetic Correlation Coefficient:** A metric that measures how faithfully a dendrogram preserves the pairwise distances between original data points.

# Python Implementation

In [1]:
import numpy as np


class AgglomerativeClustering:

  def __init__(self, n_clusters=2, linkage="single"):
    self.n_clusters = n_clusters
    self.linkage = linkage
    self.labels_ = None

  def _compute_cluster_distance(self, pts1, pts2):
    # Pairwise Euclidean distances between all points in two clusters
    dists = np.linalg.norm(pts1[:, np.newaxis] - pts2[np.newaxis, :], axis=2)
    if self.linkage == "single":
      return np.min(dists)
    elif self.linkage == "complete":
      return np.max(dists)
    elif self.linkage == "average":
      return np.mean(dists)

  def fit_predict(self, X):
    X = np.asarray(X, dtype=np.float64)
    n_samples = X.shape[0]

    # Initialize each point as its own cluster
    clusters = {i: [i] for i in range(n_samples)}

    # Build initial pairwise distance matrix
    dist_matrix = np.full((n_samples, n_samples), np.inf)
    for i in range(n_samples):
      for j in range(i + 1, n_samples):
        dist_matrix[i, j] = np.linalg.norm(X[i] - X[j])
        dist_matrix[j, i] = dist_matrix[i, j]

    # Iteratively merge closest clusters
    while len(clusters) > self.n_clusters:
      # Find pair with minimum distance
      c1_idx, c2_idx = np.unravel_index(
          np.argmin(dist_matrix), dist_matrix.shape
      )

      # Merge c2 into c1
      clusters[c1_idx].extend(clusters[c2_idx])
      del clusters[c2_idx]

      # Update distance matrix rows and columns
      dist_matrix[c2_idx, :] = np.inf
      dist_matrix[:, c2_idx] = np.inf

      for i in clusters:
        if i == c1_idx:
          continue
        d = self._compute_cluster_distance(X[clusters[c1_idx]], X[clusters[i]])
        dist_matrix[c1_idx, i] = d
        dist_matrix[i, c1_idx] = d

    # Assign cluster labels
    self.labels_ = np.zeros(n_samples, dtype=int)
    for label, (cluster_id, indices) in enumerate(clusters.items()):
      for idx in indices:
        self.labels_[idx] = label

    return self.labels_


# TEST SCRIPT

X_train = np.array([[1.0], [2.0], [6.0], [8.0]])

model = AgglomerativeClustering(n_clusters=2, linkage="single")
labels = model.fit_predict(X_train)

print("=== HIERARCHICAL CLUSTERING RESULTS ===")
for i, point in enumerate(X_train):
  print(f"Point {point[0]} -> Assigned Cluster: {labels[i]}")

=== HIERARCHICAL CLUSTERING RESULTS ===
Point 1.0 -> Assigned Cluster: 0
Point 2.0 -> Assigned Cluster: 0
Point 6.0 -> Assigned Cluster: 1
Point 8.0 -> Assigned Cluster: 1
